In [1]:

from torch import Tensor, nn
import torch
import numpy as np


# Network hyperparam
dim = 65
L = 4
mode = 20
width = 32
du = 3  # 2 for velociy x, y and pressure.

NU_LAYER = 3
da = du + 1

# NN
# Fourier Convolution Operator
class FCO(nn.Module):
    def __init__(self, mode: int, width: int) -> None:
        super().__init__()
        self.mode: int = mode
        
        self.R = nn.Parameter(torch.randn(mode, width, width, dtype=torch.complex128))

    def forward(self, x: Tensor) -> Tensor:
        # x of shape: (B, dim, dim, width)
        shape = x.shape
        x_hat = torch.fft.fftn(x, dim=(1, 2))[:, :self.mode, :self.mode, :]
        x_hat = torch.einsum('mij,bmni->bmnj', self.R, x_hat)
        x_hat = torch.fft.ifftn(x_hat, s=shape)
        x = x_hat.real
        return x


class PINOLayer(nn.Module):
    def __init__(self, mode: int, width: int) -> None:
        super().__init__()

        self.k: nn.Module = FCO(mode, width)
        self.w: nn.Module = nn.Linear(width, width, dtype=torch.float64)
        self.sigma: nn.Module = nn.GELU()

    def forward(self, x: Tensor) -> Tensor:
        w = self.w(x)
        k = self.k(x)
        x = self.sigma(w + k)
        return x


class PINO(nn.Module):
    def __init__(self, da: int, du: int, mode: int, width: int, L: int, T: int) -> None:
        super().__init__()
        self.T = T
        # uplift
        self.P = nn.Linear(da, width, dtype=torch.float64)
        # projection
        self.Q = nn.Linear(width, du, dtype=torch.float64)

        layers = [PINOLayer(mode, width) for _ in range(L)]
        self.core: nn.Module = nn.Sequential(
            self.P,
            *layers,
            self.Q,
        )

    def one_pass(self, x: Tensor) -> Tensor:
        return self.core(x)
    
    def forward(self, x: Tensor) -> Tensor:
        curr_x = x
        acc_list = []
        for i in range(self.T):
            next_x = self.one_pass(curr_x)
            acc_list.append(next_x.unsqueeze(1))  #  unsqueeze to add time dimension
            curr_x = torch.cat([next_x[..., :3], curr_x[..., 3:]], dim=-1)

        out = torch.cat(acc_list, dim=1)
        return out
            


# PDE param
T = (5, 10)
dimT = 10
alpha = 1
beta = 1

# re & nu not fixed

k = 1
dt = (T[1] - T[0]) / dimT
dx = 1.0 / dim

# Todo: fill in the choosen times
t = torch.arange(*T, dt)
x = torch.arange(0.0, 1.0, dx)
y = torch.arange(0.0, 1.0, dx)

pde_influence = 1e-3
# Training is done over elements of shape:
# batch, dimT, dim, dim, C
# We have to compute dimT elements before applying loss because pde loss needs it
# We could apply data loss more often but it would be a different scheme than L = Ldata + λLpde
# The batch dimension mixes data from different 

# To compute pde loss, we need pde params
# Dataset of shape a -> u
# Where a: (batch, dimT, dim, dim, C) with C = da (ex: du + broadcasted pde_params)
# u: (batch, dimT, dim, dim, C) with C = du

L2 = torch.nn.MSELoss(reduction="mean")


# a of shape (batch, dimT, dim, dim, da)
# PDE param embedded in da
# u  of shape (batch, dimT, dim, dim, du)

# residual loss over time
def residual_loss(a, u):
    # assumed density to be 1
    # v viscosity (nu)
    # du/dt = -(u.∇)u + v∇.∇u -∇p
    # R = du/dt + (u.∇)u + ∇p - v∇.∇u

    
    TRS = np.s_[:, 2:]  # Time restriction
    
    INT = np.s_[:, :, 1:-1, 1:-1, :]  # Interior
    XPO = np.s_[:, :,   2:, 1:-1, :]  # X Plus One
    XMO = np.s_[:, :,  :-2, 1:-1, :]  # X Minus One
    YPO = np.s_[:, :, 1:-1,   2:, :]  # Y Plus One
    YMO = np.s_[:, :, 1:-1,  :-2, :]  # Y Minus One 

    u_vel = u[..., :2]
    u_vel_t = u_vel[TRS]
    p_t = u[TRS][..., 2:2 + 1]
    
    nu = a[None][INT][..., NU_LAYER:NU_LAYER + 1]
    
    du_dt = (u_vel[INT][:, 2:] - u_vel[INT][:, 1:-1]) / dt
    
    du_dx = (u_vel_t[XPO] - u_vel_t[XMO]) / dx
    du_dy = (u_vel_t[YPO] - u_vel_t[YMO]) / dx
    du_dx2 = (u_vel_t[XPO] + u_vel_t[XMO] - 2 * u_vel_t[INT]) / (dx * dx)
    du_dy2 = (u_vel_t[YPO] + u_vel_t[YMO] - 2 * u_vel_t[INT]) / (dx * dx)
    dp_dx = (p_t[XPO] - p_t[XMO]) / dx
    dp_dy = (p_t[YPO] - p_t[YMO]) / dx
    
    c0 = u_vel_t[INT][..., 0] * du_dx[..., 0] + u_vel_t[INT][..., 1] * du_dy[..., 0]
    c1 = u_vel_t[INT][..., 0] * du_dx[..., 1] + u_vel_t[INT][..., 1] * du_dy[..., 1]
    convection = torch.stack([c0, c1], dim=-1)
    
    viscosity = nu * (du_dx2 + du_dy2)

    grad_p = torch.cat([dp_dx, dp_dy], dim=-1)
    
    R = du_dt + convection + grad_p - viscosity

    return torch.mean(R**2)

def batched_bcu(u):
    K = 1
    # top - slip
    u[..., 1:-1, 0, 0] = 2 * K - u[..., 1:-1, 1, 0]
    u[..., 1:-1, 0, 1] = - u[..., 1:-1, 1, 1]
    # bot - no slip
    u[..., 1:-1, -1, :] = - u[..., 1:-1, -2, :]
    # left - no slip
    u[..., 0, 1:-1, :] = - u[..., 1, 1:-1, :]
    # right - no slip
    u[..., -1, 1:-1, :] = - u[..., -2, 1:-1, :]


# boundary condition loss over time
def bc_loss(u):
    # compute BC
    
    u_vel = u[..., :2]
    bcu = u_vel.clone().detach()
    batched_bcu(bcu)
    return (
        L2(u_vel[:, :, 1:-1,    0, :], bcu[:, :, 1:-1,    0, :]) +
        L2(u_vel[:, :, 1:-1,   -1, :], bcu[:, :, 1:-1,   -1, :]) +
        L2(u_vel[:, :,    0, 1:-1, :], bcu[:, :,    0, 1:-1, :]) +
        L2(u_vel[:, :,   -1, 1:-1, :], bcu[:, :,   -1, 1:-1, :])
    )

# initial condition loss
def ic_loss(a, u):
    ic_vel = a[:, :, :, :2]
    u0_vel = u[:, 0, :, :, :2]
    return L2(ic_vel, u0_vel)

# L pde is J pde as long as loss is averaged over batch
def L_pde(a, pred):
    return residual_loss(a, pred) + alpha * bc_loss(pred) + beta * ic_loss(a, pred)


def full_loss(a, u, pred):
    return L2(u, pred) + pde_influence * L_pde(a, pred) 

In [2]:
# pino = PINO(da, du, mode, width, L, dimT)

In [3]:
# Batch = 1
# test_input = torch.rand(Batch, dim, dim, da, dtype=torch.float64)

In [4]:
# test_input.shape

In [5]:
# output = pino(test_input)

In [6]:
# output.shape

In [7]:
# loss = L_pde(test_input, output)

In [8]:
# loss.backward()

In [9]:
# loss

In [10]:
import os
import zarr
import s3fs
from torch.utils.data import Dataset
import torch
import torch.nn.functional as F


with open(os.path.expanduser("~") + "/.keys") as f:
    for line in f:
        if line.startswith("export "):
            key, val = line.strip().split("=", 1)
            os.environ[key.replace("export ","")] = val
fs = s3fs.S3FileSystem(
        key=os.environ["AWS_ACCESS_KEY_ID"],
        secret=os.environ["AWS_SECRET_ACCESS_KEY"],
        client_kwargs={"endpoint_url": "http://localhost:8333"}
    )

class LDCDataset(Dataset):
    def __init__(self, resolution, dimT):
        
        store_X = s3fs.S3Map(root="ldcdataset/X", s3=fs, check=False)
        store_Y = s3fs.S3Map(root="ldcdataset/Y", s3=fs, check=False)
        dts_X = zarr.open_array(store_X, mode="r")
        dts_Y = zarr.open_array(store_Y, mode="r")
        x = torch.from_numpy(dts_X[:])

        x = x.permute(0, 3, 1, 2)  # B C X X
        x = F.interpolate(x, size=(65,65), mode="bilinear", align_corners=False)
        x = x.permute(0, 2, 3, 1)  # B X X C
        self.X = x
        
        y = torch.from_numpy(dts_Y[:, :dimT])

        B, T, X, _, C = y.shape

        y = y.permute(0,1,4,2,3)  # B T C X X
        y = y.reshape(B*T, C, X, X)
        
        y = F.interpolate(y, size=(65,65), mode="bilinear", align_corners=False)
        
        y = y.reshape(B, T, C, 65, 65)
        y = y.permute(0,1,3,4,2)  # B T 65 65 C

        self.Y = y
        self.length = self.X.shape[0]

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        x = self.X[idx]
        y = self.Y[idx]
        return x, y

In [11]:
import io
def log_model(model, path, name):
    # fs.mkdirs(s3_path, exist_ok=True)
    # fs.touch(s3_path + ".keep")
    buffer = io.BytesIO()
    torch.save(model.state_dict(), buffer)
    buffer.seek(0)
    with fs.open(path + name, "wb") as f:
        f.write(buffer.read())

In [12]:
import torch
from torch import nn
from torch.optim import AdamW
from torch.amp import autocast, GradScaler
from torch.utils.data import DataLoader, random_split
from tqdm.notebook import tqdm
import trackio


def train_epoch(model, loader):
    model.train()
    total = 0.0
    pbar = tqdm(loader, leave=False)
    for x, y in pbar:
        x, y = x.cuda(non_blocking=True), y.cuda(non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        loss = criterion(x, y, model(x))
        
        loss.backward()
        optimizer.step()
        total += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4e}")
    return total / len(loader)

@torch.no_grad()
def test_epoch(model, loader):
    model.eval()
    total = 0.0
    pbar = tqdm(loader, leave=False)
    for x, y in pbar:
        x, y = x.cuda(non_blocking=True), y.cuda(non_blocking=True)
        
        loss = criterion(x, y, model(x))
        
        total += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4e}")
        
    return total / len(loader)



lr = 1e-3
epochs = 30
batch = 1

model = PINO(da, du, mode, width, L, dimT).cuda()

# with fs.open("s3://ldc-pino-models/artifacts/eager-mountain-3/final_model", "rb") as f:
#     state = torch.load(f, map_location="cpu")
# model.load_state_dict(state)

optimizer = AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
criterion = full_loss

resolution = 65
dataset = LDCDataset(resolution, dimT)
train_len = int(0.8 * len(dataset))
test_len = len(dataset) - train_len
train_dataset, test_dataset = random_split(dataset, [train_len, test_len])

train_loader = DataLoader(train_dataset, batch_size=batch, shuffle=True, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=batch, shuffle=False, num_workers=4, pin_memory=True)


params = {
    "re": 500,
    "dataset_size": len(train_dataset),
    "epochs": epochs,
    "learning_rate": lr,
    "batch_size": batch,
    "pino_mode": mode,
    "pino_L_blocks": L,
    "pino_width": width,
    "resolution": resolution,
    "pde_influence": pde_influence,
}
run = trackio.init(
    project="LDC Pino",
    config=params,
    auto_log_gpu=True,
)

* Trackio project initialized: LDC Pino
* Trackio metrics logged to: /home/kz/.cache/huggingface/trackio
* Created new run: witty-tree-17


In [13]:
s3_path = f"s3://ldc-pino-models/artifacts/{run.name}/"
save_model_interval = 10
pbar = tqdm(range(epochs), leave=True)
for epoch in pbar:
    train_loss = train_epoch(model, train_loader)
    test_loss  = test_epoch(model, test_loader)
    sched.step()
    
    pbar.set_description(f"Epoch {epoch}: train_loss={train_loss:.4e}, test_loss={test_loss:.4e}")
    trackio.log({
            "epoch": epoch,
            "train_loss": train_loss,
            "test_loss": test_loss,
        },
    )
    if epoch % save_model_interval == 0:
        log_model(model, s3_path, f"checkpoint_{epoch}")
        
trackio.finish()
log_model(model, s3_path,"final_model")

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/80 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/80 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

...

  0%|          | 0/80 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/80 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

* Run finished. Uploading logs to Trackio (please wait...)


In [14]:
x, y = next(iter(test_loader))
x, y = x.cuda(non_blocking=True), y.cuda(non_blocking=True)

In [15]:
# with fs.open("s3://ldc-pino-models/artifacts/eager-mountain-3/final_model", "rb") as f:
#     state = torch.load(f, map_location="cpu")
# model.load_state_dict(state)
# model.eval()

In [16]:
with torch.no_grad():
    out = model(x)

In [17]:
import napari

In [18]:
viewer = napari.Viewer()
viewer.add_image(y[0].cpu(), name="y")
viewer.add_image(out[0].cpu(), name="out")
viewer.add_image((abs(out - y))[0].cpu(), name="diff")
napari.run()

In [19]:
y.shape

torch.Size([1, 10, 65, 65, 3])